# RFP 파싱 및 전처리 파이프라인

현재 RFP 프로젝트를 위한 단일 전처리 파이프라인 노트북입니다.

구성 범위:

1. HWP 파싱 후보 비교
2. PDF 파싱 후보 비교
3. `data_list.csv` 메타데이터 점검 및 유용 필드 선정
4. 표와 시각자료를 위한 2차 enrichment 규칙 정의
5. 통합 `parse_document(file_path)` 파이프라인 정의

현재 파싱 정책:

- HWP 기본 파서: `hwp5txt`
- HWP 폴백: `olefile + PrvText`
- PDF 기본 파서: `pdfplumber`
- PDF 폴백: `PyMuPDF`
- 표 / 그림 / 차트에 대한 구조 신호를 보존
- 가능한 표는 텍스트 블록으로 직렬화
- 모든 이미지를 OCR하지 않고, 가치가 높은 시각자료만 선별하여 OCR 후보로 관리


In [ ]:
import json
import html as html_lib
import re
import subprocess
import tempfile
from collections import Counter
from pathlib import Path

import fitz
import numpy as np
import olefile
import pandas as pd
import pdfplumber

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data_list.csv"
FILE_DIR = BASE_DIR / "files" / "files"

HWP_FILES = sorted(FILE_DIR.glob("*.hwp"))
PDF_FILES = sorted(FILE_DIR.glob("*.pdf"))

print("기준 경로:", BASE_DIR)
print("DATA_PATH 존재 여부:", DATA_PATH.exists())
print("FILE_DIR 존재 여부:", FILE_DIR.exists())
print("HWP 파일 수:", len(HWP_FILES))
print("PDF 파일 수:", len(PDF_FILES))


## 1. HWP 파싱 전략

현재 HWP 파일의 관찰 특성:

- OLE compound file 형식이다.
- `FileHeader`, `BodyText`, `PrvText`, `BinData` 스트림을 포함한다.
- `PrvText`는 폴백용 미리보기 텍스트로 유용하다.
- `BinData`에는 표, 그림, 차트 같은 바이너리 자산이 다수 포함될 수 있다.

실무 적용 정책:

- 기본 텍스트 추출은 `hwp5txt`를 사용한다.
- `hwp5txt` 실패 시 `PrvText`를 폴백으로 사용한다.
- 구조 신호는 별도로 기록한다.
  - 추출 텍스트 내 표 / 그림 / 차트 마커
  - `BinData` 개수와 확장자 분포
- `hwp5html`은 선택적 2차 표 추출용 probe로만 사용한다. 속도가 느릴 수 있다.


In [ ]:
TABLE_MARKER_PATTERNS = [r"<표>", r"<\?\?>", r"\[TABLE\]"]
FIGURE_MARKER_PATTERNS = [r"<그림>", r"<洹몃┝>", r"\[FIGURE\]"]
CHART_MARKER_PATTERNS = [r"<차트>", r"<李⑦듃>", r"\[CHART\]"]


def run_command(cmd, timeout=60):
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="ignore",
        timeout=timeout,
    )
    return {
        "cmd": cmd,
        "returncode": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }


def clean_text(text: str) -> str:
    text = str(text).replace("\x00", " ")
    text = text.replace("\u3000", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def count_structure_markers(text: str) -> dict:
    return {
        "table_markers": sum(len(re.findall(pattern, text)) for pattern in TABLE_MARKER_PATTERNS),
        "figure_markers": sum(len(re.findall(pattern, text)) for pattern in FIGURE_MARKER_PATTERNS),
        "chart_markers": sum(len(re.findall(pattern, text)) for pattern in CHART_MARKER_PATTERNS),
    }


def read_hwp_preview_text(file_path: Path) -> str:
    if not olefile.isOleFile(str(file_path)):
        return ""
    ole = olefile.OleFileIO(str(file_path))
    try:
        if not ole.exists("PrvText"):
            return ""
        raw = ole.openstream("PrvText").read()
        return clean_text(raw.decode("utf-16le", errors="ignore"))
    finally:
        ole.close()


def inspect_hwp_ole(file_path: Path) -> dict:
    info = {
        "is_ole": False,
        "stream_count": 0,
        "has_fileheader": False,
        "has_prvtext": False,
        "has_bodytext": False,
        "bin_count": 0,
        "bin_ext_counts": {},
    }
    if not olefile.isOleFile(str(file_path)):
        return info

    ole = olefile.OleFileIO(str(file_path))
    try:
        streams = ole.listdir()
        info["is_ole"] = True
        info["stream_count"] = len(streams)
        info["has_fileheader"] = ole.exists("FileHeader")
        info["has_prvtext"] = ole.exists("PrvText")
        info["has_bodytext"] = any(parts[0] == "BodyText" for parts in streams if parts)

        bin_names = [parts[-1] for parts in streams if len(parts) >= 2 and parts[0] == "BinData"]
        ext_counter = Counter()
        for name in bin_names:
            ext_counter[Path(name).suffix.lower() or "<no_ext>"] += 1

        info["bin_count"] = len(bin_names)
        info["bin_ext_counts"] = dict(ext_counter)
        return info
    finally:
        ole.close()


def extract_hwp_text_hwp5txt(file_path: Path) -> dict:
    result = run_command(["hwp5txt", str(file_path)], timeout=120)
    text = clean_text(result["stdout"])
    return {
        "text": text,
        "returncode": result["returncode"],
        "stderr_head": result["stderr"][:1000],
        "text_len": len(text),
    }


def resolve_generated_html_path(output_target: Path, temp_root: Path):
    candidates = []
    if output_target.exists():
        if output_target.is_file():
            candidates.append(output_target)
        elif output_target.is_dir():
            candidates.extend(output_target.rglob("*.html"))
    candidates.extend(temp_root.rglob("*.html"))
    candidates = [path for path in candidates if path.exists()]
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_size)


def extract_hwp_html_summary(file_path: Path, timeout=20) -> dict:
    summary = {
        "success": False,
        "html_table_count": None,
        "html_img_count": None,
        "html_len": None,
        "html_path": None,
        "error": None,
    }
    with tempfile.TemporaryDirectory() as td:
        temp_root = Path(td)
        output_target = temp_root / "preview"
        try:
            result = run_command(["hwp5html", "--output", str(output_target), str(file_path)], timeout=timeout)
            html_path = resolve_generated_html_path(output_target, temp_root)
            if result["returncode"] != 0 or html_path is None:
                summary["error"] = result["stderr"][:500]
                return summary

            html = html_path.read_text(encoding="utf-8", errors="ignore")
            summary["success"] = True
            summary["html_table_count"] = len(re.findall(r"<table\b", html, re.I))
            summary["html_img_count"] = len(re.findall(r"<img\b", html, re.I))
            summary["html_len"] = len(html)
            summary["html_path"] = str(html_path)
            return summary
        except Exception as exc:
            summary["error"] = str(exc)
            return summary


def compare_hwp_parsers(file_path: Path, run_html=False) -> dict:
    hwp5 = extract_hwp_text_hwp5txt(file_path)
    preview = read_hwp_preview_text(file_path)
    ole_info = inspect_hwp_ole(file_path)
    html_info = extract_hwp_html_summary(file_path) if run_html else {"success": False}
    return {
        "file_name": file_path.name,
        "hwp5txt_text_len": hwp5["text_len"],
        "hwp5txt_markers": count_structure_markers(hwp5["text"]),
        "hwp5txt_stderr_head": hwp5["stderr_head"],
        "prvtext_len": len(preview),
        "prvtext_markers": count_structure_markers(preview),
        "ole_info": ole_info,
        "html_info": html_info,
        "hwp5txt_preview": hwp5["text"][:1200],
        "prvtext_preview": preview[:1200],
    }


In [ ]:
sample_hwp = HWP_FILES[0]
hwp_compare = compare_hwp_parsers(sample_hwp, run_html=False)

print("샘플 HWP:", sample_hwp.name)
print("hwp5txt 추출 길이:", hwp_compare["hwp5txt_text_len"])
print("PrvText 길이:", hwp_compare["prvtext_len"])
print("OLE 정보:", json.dumps(hwp_compare["ole_info"], ensure_ascii=False, indent=2))
print("hwp5txt 마커 수:", hwp_compare["hwp5txt_markers"])
print("hwp5txt 미리보기:")
print(hwp_compare["hwp5txt_preview"])


### HWP 결론

현재 데이터셋에서는 `hwp5txt`가 기본 HWP 파서로 적절하다.

이유:

- `PrvText`보다 훨씬 많은 텍스트를 추출한다.
- 표/그림 관련 구조 placeholder를 어느 정도 보존한다.
- 요구사항, 예산, 일정, 계약 관련 텍스트를 baseline 수준에서 확보하기에 충분하다.

최종 HWP 정책:

- 기본: `hwp5txt`
- 폴백: `olefile + PrvText`
- `BinData` 인벤토리와 구조 마커 수를 메타데이터에 유지
- `hwp5html`은 선택된 파일에 대해서만 2차 표 추출용으로 사용


## 2. PDF 파싱 전략

현재 한글 RFP PDF에 대해서는 다음 두 가지를 비교한다.

- `pdfplumber`: 줄 단위 텍스트가 더 깔끔하고 표 추출 훅이 바로 있다.
- `PyMuPDF`: 일반 텍스트 추출이 빠르고 안정적이다.

현재 적용 정책:

- 기본: `pdfplumber`
- 폴백: `PyMuPDF`
- 표 신호: `pdfplumber.extract_tables()`


In [ ]:
def extract_pdf_text_pymupdf(file_path: Path) -> dict:
    doc = fitz.open(file_path)
    try:
        pages = [page.get_text("text") for page in doc]
        text = clean_text("\n\n".join(pages))
        return {"text": text, "page_count": len(doc), "text_len": len(text)}
    finally:
        doc.close()


def extract_pdf_text_pdfplumber(file_path: Path) -> dict:
    pages = []
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            pages.append(page.extract_text() or "")
    text = clean_text("\n\n".join(pages))
    return {"text": text, "page_count": len(pages), "text_len": len(text)}


def extract_pdf_table_summary_pdfplumber(file_path: Path, max_pages=5) -> dict:
    page_table_counts = []
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages[:max_pages]:
            try:
                tables = page.extract_tables() or []
                page_table_counts.append(len(tables))
            except Exception:
                page_table_counts.append(0)
    return {
        "checked_pages": len(page_table_counts),
        "table_count_first_pages": int(sum(page_table_counts)),
        "page_table_counts": page_table_counts,
    }


def compare_pdf_parsers(file_path: Path) -> dict:
    plumber = extract_pdf_text_pdfplumber(file_path)
    pymupdf = extract_pdf_text_pymupdf(file_path)
    table_summary = extract_pdf_table_summary_pdfplumber(file_path)
    return {
        "file_name": file_path.name,
        "pdfplumber_text_len": plumber["text_len"],
        "pymupdf_text_len": pymupdf["text_len"],
        "page_count_plumber": plumber["page_count"],
        "page_count_pymupdf": pymupdf["page_count"],
        "table_summary": table_summary,
        "pdfplumber_preview": plumber["text"][:1200],
        "pymupdf_preview": pymupdf["text"][:1200],
    }


In [ ]:
if PDF_FILES:
    sample_pdf = PDF_FILES[0]
    pdf_compare = compare_pdf_parsers(sample_pdf)
    print("샘플 PDF:", sample_pdf.name)
    print("pdfplumber 추출 길이:", pdf_compare["pdfplumber_text_len"])
    print("PyMuPDF 추출 길이:", pdf_compare["pymupdf_text_len"])
    print("표 요약:", json.dumps(pdf_compare["table_summary"], ensure_ascii=False, indent=2))
    print("pdfplumber 미리보기:")
    print(pdf_compare["pdfplumber_preview"])
    print("\nPyMuPDF_preview:")
    print(pdf_compare["pymupdf_preview"])
else:
    print("PDF 파일이 없습니다.")


### PDF 결론

현재 프로젝트에서는 `pdfplumber`를 기본값으로 두는 편이 더 적절하다.

이유:

- 오피스 기반 한글 PDF에서 텍스트 레이아웃이 더 깔끔한 경우가 많다.
- 표 추출 훅을 바로 활용할 수 있다.

최종 PDF 정책:

- 기본: `pdfplumber`
- 폴백: `PyMuPDF`
- PDF 표 개수를 메타데이터에 유지


## 3. `data_list.csv` 메타데이터 분석

retrieval, filtering, 비교 질의에 직접적으로 도움이 되는 필드만 남기는 것이 목적이다.


In [ ]:
def load_csv_with_fallback(path: Path) -> pd.DataFrame:
    encodings = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as exc:
            last_error = exc
    raise last_error


metadata_df = load_csv_with_fallback(DATA_PATH)
metadata_df.columns = [str(column).strip() for column in metadata_df.columns]
metadata_columns = list(metadata_df.columns)
print(metadata_columns)

profile_rows = []
for col in metadata_df.columns:
    series = metadata_df[col]
    non_null = series.notna().sum()
    profile_rows.append(
        {
            "column": col,
            "dtype": str(series.dtype),
            "non_null": int(non_null),
            "missing": int(len(series) - non_null),
            "missing_ratio_pct": round((len(series) - non_null) / len(series) * 100, 2),
            "nunique": int(series.nunique(dropna=True)),
        }
    )

metadata_profile_df = pd.DataFrame(profile_rows).sort_values(["missing", "nunique"])
display(metadata_profile_df)


In [ ]:
# 현재 CSV의 예상 컬럼 순서:
# 0 announcement_no, 1 announcement_round, 2 project_name, 3 budget,
# 4 agency, 5 posted_at, 6 bid_start, 7 bid_end, 8 project_summary,
# 9 file_type, 10 file_name, 11 full_text

COL_ANNOUNCE_NO = metadata_columns[0]
COL_ANNOUNCE_ROUND = metadata_columns[1]
COL_PROJECT_NAME = metadata_columns[2]
COL_BUDGET = metadata_columns[3]
COL_AGENCY = metadata_columns[4]
COL_POSTED_AT = metadata_columns[5]
COL_BID_START = metadata_columns[6]
COL_BID_END = metadata_columns[7]
COL_PROJECT_SUMMARY = metadata_columns[8]
COL_FILE_TYPE = metadata_columns[9]
COL_FILE_NAME = metadata_columns[10]
COL_FULL_TEXT = metadata_columns[11]

selected_metadata_fields = [
    COL_PROJECT_NAME,
    COL_AGENCY,
    COL_BUDGET,
    COL_POSTED_AT,
    COL_BID_START,
    COL_BID_END,
    COL_FILE_TYPE,
    COL_FILE_NAME,
    COL_ANNOUNCE_NO,
    COL_ANNOUNCE_ROUND,
]

print("선정 메타데이터 필드:", selected_metadata_fields)
display(metadata_df[selected_metadata_fields].head(5))


### 메타데이터 결론

전처리 파이프라인에는 다음 필드를 유지한다.

- 사업명
- 발주기관
- 예산
- 공고일
- 입찰 참여 시작일 / 마감일
- 파일 형식
- 파일명
- 공고 번호 / 차수(선택적 식별자)


## 4. 표와 시각자료를 위한 2차 enrichment 설계

기본 파서는 marker만 남기기 때문에, 사용자가 표 내용이나 차트 내용을 물을 경우에는 충분하지 않다.

2차 정책:

- 표:
  - 추출 가능한 표는 markdown 유사 텍스트 블록으로 변환
  - retrieval corpus에 함께 추가
- 시각자료:
  - 모든 이미지를 무조건 OCR하지 않음
  - 먼저 heuristic shortlist를 만든 뒤
  - 우선순위가 높은 후보만 OCR 수행
  - 사람 검토는 shortlist 되거나 애매한 경우에만 사용


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".gif", ".bmp", ".tif", ".tiff", ".emf", ".wmf"}
VISUAL_REFERENCE_PATTERNS = [
    r"아래\s*(그림|도표|차트|구성도|흐름도)",
    r"다음\s*(그림|도표|차트|구성도|흐름도)",
    r"(시스템|서비스)?\s*구성도",
    r"아키텍처",
    r"흐름도",
    r"프로세스",
    r"화면\s*예시",
    r"예시\s*화면",
    r"대시보드",
    r"차트",
    r"그래프",
    r"도식",
    r"ERD",
]
HIGH_VALUE_VISUAL_PATTERNS = [r"구성도", r"아키텍처", r"흐름도", r"프로세스", r"ERD", r"화면\s*예시", r"대시보드", r"차트", r"그래프"]
LOW_VALUE_VISUAL_PATTERNS = [r"로고", r"직인", r"서명", r"배경", r"장식"]


def escape_markdown_cell(value) -> str:
    value = clean_text(str(value or ""))
    value = value.replace("|", "/")
    return value


def rows_to_markdown_table(rows, max_rows=12, max_cols=8) -> str:
    cleaned_rows = []
    for row in rows or []:
        normalized = [escape_markdown_cell(cell) for cell in list(row)[:max_cols]]
        if any(cell for cell in normalized):
            cleaned_rows.append(normalized)
    if not cleaned_rows:
        return ""

    cleaned_rows = cleaned_rows[:max_rows]
    width = max(max(len(row) for row in cleaned_rows), 1)
    padded_rows = [row + [""] * (width - len(row)) for row in cleaned_rows]
    use_first_row_as_header = len(padded_rows) >= 2 and sum(bool(cell) for cell in padded_rows[0]) >= max(1, width // 2)

    if use_first_row_as_header:
        header = padded_rows[0]
        body_rows = padded_rows[1:]
    else:
        header = [f"col_{idx + 1}" for idx in range(width)]
        body_rows = padded_rows

    lines = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * width) + " |"]
    lines.extend("| " + " | ".join(row) + " |" for row in body_rows)
    return "\n".join(lines)


def html_fragment_to_text(fragment: str) -> str:
    text = re.sub(r"<br\s*/?>", "\n", fragment, flags=re.I)
    text = re.sub(r"</(p|div|tr|li|h\d)>", "\n", text, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    text = html_lib.unescape(text)
    return clean_text(text)


def extract_tables_from_html(html_text: str, max_tables=5) -> list[dict]:
    table_blocks = []
    table_matches = re.findall(r"(<table\b.*?</table>)", html_text, flags=re.I | re.S)
    for table_idx, table_html in enumerate(table_matches[:max_tables], start=1):
        row_matches = re.findall(r"<tr\b.*?>(.*?)</tr>", table_html, flags=re.I | re.S)
        rows = []
        for row_html in row_matches:
            cells = re.findall(r"<t[dh]\b.*?>(.*?)</t[dh]>", row_html, flags=re.I | re.S)
            normalized = [html_fragment_to_text(cell) for cell in cells]
            if any(normalized):
                rows.append(normalized)

        markdown = rows_to_markdown_table(rows)
        if markdown:
            table_blocks.append({"table_index": table_idx, "source": "hwp5html", "row_count": len(rows), "markdown": markdown})
    return table_blocks


In [ ]:
def extract_hwp_rich_structure_hwp5html(file_path: Path, timeout=30, max_tables=5) -> dict:
    summary = {
        "success": False,
        "html_table_count": None,
        "html_img_count": None,
        "html_len": None,
        "table_blocks": [],
        "img_sources_preview": [],
        "error": None,
    }

    with tempfile.TemporaryDirectory() as td:
        temp_root = Path(td)
        output_target = temp_root / "preview"
        try:
            result = run_command(["hwp5html", "--output", str(output_target), str(file_path)], timeout=timeout)
            html_path = resolve_generated_html_path(output_target, temp_root)
            if result["returncode"] != 0 or html_path is None:
                summary["error"] = result["stderr"][:500]
                return summary

            html_text = html_path.read_text(encoding="utf-8", errors="ignore")
            img_sources = re.findall(r"<img\b[^>]*src=[\"']([^\"']+)[\"']", html_text, flags=re.I)

            summary["success"] = True
            summary["html_table_count"] = len(re.findall(r"<table\b", html_text, flags=re.I))
            summary["html_img_count"] = len(re.findall(r"<img\b", html_text, flags=re.I))
            summary["html_len"] = len(html_text)
            summary["table_blocks"] = extract_tables_from_html(html_text, max_tables=max_tables)
            summary["img_sources_preview"] = img_sources[:10]
            return summary
        except Exception as exc:
            summary["error"] = str(exc)
            return summary


def extract_pdf_tables_as_markdown(file_path: Path, max_pages=5, max_tables_per_page=3) -> list[dict]:
    table_blocks = []
    with pdfplumber.open(file_path) as pdf:
        for page_number, page in enumerate(pdf.pages[:max_pages], start=1):
            try:
                tables = page.extract_tables() or []
            except Exception:
                tables = []

            for table_idx, rows in enumerate(tables[:max_tables_per_page], start=1):
                markdown = rows_to_markdown_table(rows)
                if markdown:
                    table_blocks.append(
                        {
                            "page_number": page_number,
                            "table_index": table_idx,
                            "source": "pdfplumber",
                            "row_count": len(rows or []),
                            "markdown": markdown,
                        }
                    )
    return table_blocks


def extract_reference_snippets(text: str, patterns=None, window=120, max_snippets=8) -> list[dict]:
    patterns = patterns or VISUAL_REFERENCE_PATTERNS
    hits = []
    for pattern in patterns:
        for match in re.finditer(pattern, text, flags=re.I):
            start = max(0, match.start() - window)
            end = min(len(text), match.end() + window)
            hits.append(
                {
                    "match_text": clean_text(match.group(0)),
                    "snippet": clean_text(text[start:end]),
                    "start": match.start(),
                    "end": match.end(),
                }
            )

    deduped = []
    seen = set()
    for item in sorted(hits, key=lambda row: (row["start"], row["end"])):
        key = (item["match_text"], item["snippet"])
        if key in seen:
            continue
        seen.add(key)
        deduped.append(item)
        if len(deduped) >= max_snippets:
            break
    return deduped


def count_image_like_assets(bin_ext_counts: dict) -> int:
    return int(sum(count for ext, count in (bin_ext_counts or {}).items() if str(ext).lower() in IMAGE_EXTS))


def recommend_visual_action(score: int) -> str:
    if score >= 5:
        return "ocr_priority_then_manual_check"
    if score >= 3:
        return "ocr_if_user_asks_visual_detail"
    return "keep_marker_only"


def score_visual_candidate(
    snippet: str,
    match_text: str = "",
    image_asset_count: int = 0,
    page_image_count: int = 0,
    figure_markers: int = 0,
    chart_markers: int = 0,
) -> tuple[int, list[str]]:
    joined = clean_text(f"{match_text} {snippet}")
    score = 0
    reasons = []

    for pattern in HIGH_VALUE_VISUAL_PATTERNS:
        if re.search(pattern, joined, flags=re.I):
            score += 2
            reasons.append(f"keyword:{pattern}")

    if figure_markers > 0:
        score += 1
        reasons.append(f"figure_markers={figure_markers}")

    if chart_markers > 0:
        score += 1
        reasons.append(f"chart_markers={chart_markers}")

    if image_asset_count > 0:
        score += min(2, image_asset_count)
        reasons.append(f"image_assets={image_asset_count}")

    if page_image_count > 0:
        score += min(2, page_image_count)
        reasons.append(f"page_images={page_image_count}")

    for pattern in LOW_VALUE_VISUAL_PATTERNS:
        if re.search(pattern, joined, flags=re.I):
            score -= 2
            reasons.append(f"low_value:{pattern}")

    return max(score, 0), reasons


In [ ]:
def build_hwp_visual_candidates(text: str, ole_info: dict, markers: dict, top_k=5) -> list[dict]:
    image_asset_count = count_image_like_assets(ole_info.get("bin_ext_counts", {}))
    snippets = extract_reference_snippets(text)
    candidates = []

    for idx, item in enumerate(snippets, start=1):
        score, reasons = score_visual_candidate(
            snippet=item["snippet"],
            match_text=item["match_text"],
            image_asset_count=image_asset_count,
            figure_markers=markers.get("figure_markers", 0),
            chart_markers=markers.get("chart_markers", 0),
        )
        candidates.append(
            {
                "candidate_id": f"hwp_ref_{idx}",
                "candidate_type": "text_reference",
                "page_number": None,
                "reference_text": item["match_text"],
                "snippet": item["snippet"],
                "score": score,
                "reasons": reasons,
                "recommended_action": recommend_visual_action(score),
            }
        )

    if not candidates and image_asset_count > 0:
        score = min(4, 1 + image_asset_count)
        candidates.append(
            {
                "candidate_id": "hwp_asset_inventory_only",
                "candidate_type": "asset_inventory_only",
                "page_number": None,
                "reference_text": None,
                "snippet": "명시적인 그림 참조 문구는 없지만 이미지형 BinData 자산이 존재합니다.",
                "score": score,
                "reasons": [f"image_assets={image_asset_count}"],
                "recommended_action": recommend_visual_action(score),
            }
        )

    candidates.sort(key=lambda row: (-row["score"], row["candidate_id"]))
    return candidates[:top_k]


def build_pdf_visual_candidates(file_path: Path, max_pages=10, top_k=5) -> list[dict]:
    candidates = []
    doc = fitz.open(file_path)
    try:
        for page_number, page in enumerate(doc[:max_pages], start=1):
            page_text = clean_text(page.get_text("text"))
            page_image_count = len(page.get_images(full=True))
            snippets = extract_reference_snippets(page_text, max_snippets=3)

            if snippets:
                for idx, item in enumerate(snippets, start=1):
                    score, reasons = score_visual_candidate(
                        snippet=item["snippet"],
                        match_text=item["match_text"],
                        page_image_count=page_image_count,
                    )
                    candidates.append(
                        {
                            "candidate_id": f"pdf_p{page_number}_ref_{idx}",
                            "candidate_type": "page_reference",
                            "page_number": page_number,
                            "reference_text": item["match_text"],
                            "snippet": item["snippet"],
                            "score": score,
                            "reasons": reasons,
                            "recommended_action": recommend_visual_action(score),
                        }
                    )
            elif page_image_count > 0:
                score = min(4, 1 + page_image_count)
                candidates.append(
                    {
                        "candidate_id": f"pdf_p{page_number}_images_only",
                        "candidate_type": "page_images_only",
                        "page_number": page_number,
                        "reference_text": None,
                        "snippet": "이 페이지에는 이미지가 포함되어 있지만 명시적인 그림 참조 문구는 없습니다.",
                        "score": score,
                        "reasons": [f"page_images={page_image_count}"],
                        "recommended_action": recommend_visual_action(score),
                    }
                )
    finally:
        doc.close()

    candidates.sort(key=lambda row: (-row["score"], row["candidate_id"]))
    return candidates[:top_k]


def build_ocr_queue(visual_candidates: list[dict], min_score=4) -> list[dict]:
    queue = []
    for candidate in visual_candidates:
        if candidate["score"] < min_score:
            continue
        queue.append(
            {
                "candidate_id": candidate["candidate_id"],
                "candidate_type": candidate["candidate_type"],
                "page_number": candidate.get("page_number"),
                "reference_text": candidate.get("reference_text"),
                "snippet": candidate.get("snippet"),
                "score": candidate["score"],
                "recommended_action": candidate["recommended_action"],
                "reasons": candidate["reasons"],
            }
        )
    return queue


def build_rag_ready_text(parsed_doc: dict, include_table_blocks=True, include_visual_hints=False, max_tables=3, max_visual_hints=2) -> str:
    parts = [parsed_doc["text"]]
    artifacts = parsed_doc.get("artifacts", {})

    if include_table_blocks:
        for block in artifacts.get("table_blocks", [])[:max_tables]:
            prefix = f"[TABLE BLOCK][source={block['source']}]"
            if block.get("page_number") is not None:
                prefix += f"[page={block['page_number']}]"
            parts.append(prefix + "\n" + block["markdown"])

    if include_visual_hints:
        for candidate in artifacts.get("ocr_queue", [])[:max_visual_hints]:
            parts.append(
                f"[VISUAL OCR CANDIDATE][score={candidate['score']}][action={candidate['recommended_action']}]\n"
                f"{candidate.get('reference_text') or candidate['candidate_type']}\n"
                f"{candidate.get('snippet') or ''}"
            )

    return "\n\n".join(part for part in parts if part).strip()


### 시각자료 중요도 점수화 정책

시각자료의 중요도는 사람이 모든 이미지를 직접 보면서 정하는 방식이 아니다.

권장 절차:

1. 휴리스틱 shortlist:
   - `구성도`, `아키텍처`, `흐름도`, `프로세스`, `차트`, `그래프`, `화면 예시` 같은 주변 키워드 탐색
   - HWP `BinData` 안의 이미지형 자산 개수 확인
   - PDF 페이지별 embedded image 개수 확인
2. OCR 큐 생성:
   - 임계값 이상 후보만 OCR 대상으로 보낸다.
3. 사람 검토:
   - shortlist 된 후보만 확인
   - OCR 결과가 지저분한 경우 확인
   - 사용자가 특정 시각자료 설명을 직접 요구한 경우 확인


## 5. 통합 파이프라인

최종 목표:

```python
parse_document(file_path) -> {
    "text": ...,
    "metadata": ...,
    "parser_info": ...,
    "artifacts": ...
}
```

`metadata`에는 CSV 메타데이터와 파서가 만든 구조 메타데이터를 함께 넣는다.
`artifacts`에는 표 블록, 시각자료 후보, OCR 후보 큐를 저장한다.


In [ ]:
def coerce_metadata_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    return value


def build_metadata_lookup(df: pd.DataFrame, selected_fields: list[str]) -> dict:
    lookup = {}
    for _, row in df.iterrows():
        key = str(row[COL_FILE_NAME]).strip()
        if not key:
            continue
        lookup[key] = {
            field: coerce_metadata_value(row[field])
            for field in selected_fields
            if field in df.columns
        }
    return lookup


METADATA_LOOKUP = build_metadata_lookup(metadata_df, selected_metadata_fields)
print("metadata_lookup 크기:", len(METADATA_LOOKUP))


def parse_hwp_document(file_path: Path, metadata_lookup=None, enrich_tables=False, enrich_visuals=True, ocr_score_threshold=4) -> dict:
    file_path = Path(file_path)
    metadata_lookup = metadata_lookup or METADATA_LOOKUP

    hwp5 = extract_hwp_text_hwp5txt(file_path)
    preview = read_hwp_preview_text(file_path)
    ole_info = inspect_hwp_ole(file_path)

    text = hwp5["text"]
    parser_used = "hwp5txt"
    fallback_used = False
    if not text:
        text = preview
        parser_used = "ole_prvtext"
        fallback_used = True

    markers = count_structure_markers(text)
    csv_meta = metadata_lookup.get(file_path.name, {})

    rich_structure = {
        "success": False,
        "html_table_count": None,
        "html_img_count": None,
        "html_len": None,
        "table_blocks": [],
        "img_sources_preview": [],
        "error": None,
    }
    if enrich_tables:
        rich_structure = extract_hwp_rich_structure_hwp5html(file_path, timeout=30, max_tables=5)

    table_blocks = rich_structure["table_blocks"] if enrich_tables and rich_structure.get("success") else []
    visual_candidates = build_hwp_visual_candidates(text, ole_info, markers, top_k=5) if enrich_visuals else []
    ocr_queue = build_ocr_queue(visual_candidates, min_score=ocr_score_threshold) if enrich_visuals else []

    metadata = {
        **csv_meta,
        "source_path": str(file_path),
        "source_file_name": file_path.name,
        "source_extension": file_path.suffix.lower(),
        "bin_count": ole_info["bin_count"],
        "bin_ext_counts": ole_info["bin_ext_counts"],
        "table_markers": markers["table_markers"],
        "figure_markers": markers["figure_markers"],
        "chart_markers": markers["chart_markers"],
        "enriched_table_count": len(table_blocks),
        "visual_candidate_count": len(visual_candidates),
        "ocr_candidate_count": len(ocr_queue),
        "visual_priority_max_score": max((row["score"] for row in visual_candidates), default=0),
    }

    parser_info = {
        "file_type": "hwp",
        "parser_used": parser_used,
        "fallback_used": fallback_used,
        "text_length": len(text),
        "ole_info": ole_info,
        "html_probe": {key: value for key, value in rich_structure.items() if key != "table_blocks"},
        "stderr_head": hwp5["stderr_head"],
    }

    artifacts = {"table_blocks": table_blocks, "visual_candidates": visual_candidates, "ocr_queue": ocr_queue}
    return {"text": text, "metadata": metadata, "parser_info": parser_info, "artifacts": artifacts}


def parse_pdf_document(file_path: Path, metadata_lookup=None, preferred_parser="pdfplumber", enrich_tables=True, enrich_visuals=True, ocr_score_threshold=4) -> dict:
    file_path = Path(file_path)
    metadata_lookup = metadata_lookup or METADATA_LOOKUP

    if preferred_parser == "pdfplumber":
        primary = extract_pdf_text_pdfplumber(file_path)
        secondary = extract_pdf_text_pymupdf(file_path)
    else:
        primary = extract_pdf_text_pymupdf(file_path)
        secondary = extract_pdf_text_pdfplumber(file_path)

    text = primary["text"]
    parser_used = preferred_parser
    fallback_used = False
    if not text:
        text = secondary["text"]
        parser_used = "pymupdf" if preferred_parser == "pdfplumber" else "pdfplumber"
        fallback_used = True

    table_summary = extract_pdf_table_summary_pdfplumber(file_path)
    csv_meta = metadata_lookup.get(file_path.name, {})

    table_blocks = extract_pdf_tables_as_markdown(file_path, max_pages=5, max_tables_per_page=3) if enrich_tables else []
    visual_candidates = build_pdf_visual_candidates(file_path, max_pages=10, top_k=5) if enrich_visuals else []
    ocr_queue = build_ocr_queue(visual_candidates, min_score=ocr_score_threshold) if enrich_visuals else []

    metadata = {
        **csv_meta,
        "source_path": str(file_path),
        "source_file_name": file_path.name,
        "source_extension": file_path.suffix.lower(),
        "pdf_table_count_first_pages": table_summary["table_count_first_pages"],
        "pdf_page_table_counts": table_summary["page_table_counts"],
        "enriched_table_count": len(table_blocks),
        "visual_candidate_count": len(visual_candidates),
        "ocr_candidate_count": len(ocr_queue),
        "visual_priority_max_score": max((row["score"] for row in visual_candidates), default=0),
    }

    parser_info = {
        "file_type": "pdf",
        "parser_used": parser_used,
        "fallback_used": fallback_used,
        "text_length": len(text),
        "primary_text_len": primary["text_len"],
        "secondary_text_len": secondary["text_len"],
        "table_summary": table_summary,
    }

    artifacts = {"table_blocks": table_blocks, "visual_candidates": visual_candidates, "ocr_queue": ocr_queue}
    return {"text": text, "metadata": metadata, "parser_info": parser_info, "artifacts": artifacts}


def parse_document(file_path, metadata_lookup=None, enrich_tables=None, enrich_visuals=True, ocr_score_threshold=4) -> dict:
    file_path = Path(file_path)
    suffix = file_path.suffix.lower()

    if suffix == ".hwp":
        return parse_hwp_document(
            file_path,
            metadata_lookup=metadata_lookup,
            enrich_tables=False if enrich_tables is None else enrich_tables,
            enrich_visuals=enrich_visuals,
            ocr_score_threshold=ocr_score_threshold,
        )
    if suffix == ".pdf":
        return parse_pdf_document(
            file_path,
            metadata_lookup=metadata_lookup,
            enrich_tables=True if enrich_tables is None else enrich_tables,
            enrich_visuals=enrich_visuals,
            ocr_score_threshold=ocr_score_threshold,
        )
    raise ValueError(f"Unsupported file type: {suffix}")


In [ ]:
sample_hwp_result = parse_document(HWP_FILES[0], enrich_tables=False, enrich_visuals=True)
print("HWP 사용 파서:", sample_hwp_result["parser_info"]["parser_used"])
print("HWP 텍스트 길이:", len(sample_hwp_result["text"]))
print("HWP 메타데이터 미리보기:", json.dumps(sample_hwp_result["metadata"], ensure_ascii=False, indent=2)[:1200])
print("HWP 시각자료 후보:", json.dumps(sample_hwp_result["artifacts"]["visual_candidates"][:3], ensure_ascii=False, indent=2))

if PDF_FILES:
    sample_pdf_result = parse_document(PDF_FILES[0], enrich_tables=True, enrich_visuals=True)
    print("\nPDF parser_used:", sample_pdf_result["parser_info"]["parser_used"])
    print("PDF 텍스트 길이:", len(sample_pdf_result["text"]))
    print("PDF 메타데이터 미리보기:", json.dumps(sample_pdf_result["metadata"], ensure_ascii=False, indent=2)[:1200])
    print("PDF 표 블록 수:", len(sample_pdf_result["artifacts"]["table_blocks"]))
    print("PDF OCR 후보 큐:", json.dumps(sample_pdf_result["artifacts"]["ocr_queue"][:3], ensure_ascii=False, indent=2))


In [ ]:
sample_files = HWP_FILES[:3] + PDF_FILES[:2]
rows = []
for file_path in sample_files:
    parsed = parse_document(file_path)
    rows.append(
        {
            "file_name": file_path.name,
            "ext": file_path.suffix.lower(),
            "parser_used": parsed["parser_info"]["parser_used"],
            "text_len": len(parsed["text"]),
            "project_name": parsed["metadata"].get(COL_PROJECT_NAME),
            "agency": parsed["metadata"].get(COL_AGENCY),
            "budget": parsed["metadata"].get(COL_BUDGET),
            "table_markers": parsed["metadata"].get("table_markers"),
            "figure_markers": parsed["metadata"].get("figure_markers"),
            "bin_count": parsed["metadata"].get("bin_count"),
            "enriched_table_count": parsed["metadata"].get("enriched_table_count"),
            "visual_candidate_count": parsed["metadata"].get("visual_candidate_count"),
            "ocr_candidate_count": parsed["metadata"].get("ocr_candidate_count"),
            "visual_priority_max_score": parsed["metadata"].get("visual_priority_max_score"),
        }
    )

batch_preview_df = pd.DataFrame(rows)
display(batch_preview_df)


In [ ]:
if PDF_FILES:
    rag_ready_preview = build_rag_ready_text(
        parse_document(PDF_FILES[0], enrich_tables=True, enrich_visuals=True),
        include_table_blocks=True,
        include_visual_hints=False,
    )
    print(rag_ready_preview[:2000])
else:
    print("PDF 파일이 없습니다.")


## 6. 전체 문서 일괄 전처리 저장

앞선 셀들은 파이프라인을 정의하고 샘플 문서로 검증하는 단계이다.
다음 셀은 전체 문서를 일괄 전처리한 뒤 `processed_data/` 아래에 저장한다.

저장 산출물:

- `processed_documents.jsonl`: 문서별 전체 전처리 결과
- `processed_documents_summary.csv`: 가벼운 요약 테이블
- `processed_ocr_queue.jsonl`: OCR 대상 시각자료 후보 목록
- `preprocess_errors.jsonl`: 문서별 실패 내역
- `preprocessing_run_config.json`: 실행 설정

권장 기본값:

- 첫 전체 실행에서는 `hwp5html`이 느리므로 HWP 표 enrichment는 끄는 편이 안전하다.
- PDF 표 enrichment는 켠다.
- 시각자료 enrichment는 켜서 OCR 후보를 잃지 않도록 한다.


In [ ]:
OUTPUT_DIR = BASE_DIR / "processed_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HWP_TABLE_ENRICHMENT = False
PDF_TABLE_ENRICHMENT = True
VISUAL_ENRICHMENT = True
OCR_SCORE_THRESHOLD = 4
INCLUDE_VISUAL_HINTS_IN_RAG_TEXT = False

DOCUMENTS_JSONL_PATH = OUTPUT_DIR / "processed_documents.jsonl"
SUMMARY_CSV_PATH = OUTPUT_DIR / "processed_documents_summary.csv"
OCR_QUEUE_JSONL_PATH = OUTPUT_DIR / "processed_ocr_queue.jsonl"
ERRORS_JSONL_PATH = OUTPUT_DIR / "preprocess_errors.jsonl"
RUN_CONFIG_PATH = OUTPUT_DIR / "preprocessing_run_config.json"

all_files = HWP_FILES + PDF_FILES
processed_records = []
summary_rows = []
ocr_rows = []
error_rows = []

run_config = {
    "base_dir": str(BASE_DIR),
    "output_dir": str(OUTPUT_DIR),
    "document_count": len(all_files),
    "hwp_table_enrichment": HWP_TABLE_ENRICHMENT,
    "pdf_table_enrichment": PDF_TABLE_ENRICHMENT,
    "visual_enrichment": VISUAL_ENRICHMENT,
    "ocr_score_threshold": OCR_SCORE_THRESHOLD,
    "include_visual_hints_in_rag_text": INCLUDE_VISUAL_HINTS_IN_RAG_TEXT,
}

for idx, file_path in enumerate(all_files, start=1):
    enrich_tables = PDF_TABLE_ENRICHMENT if file_path.suffix.lower() == ".pdf" else HWP_TABLE_ENRICHMENT

    try:
        parsed = parse_document(
            file_path,
            enrich_tables=enrich_tables,
            enrich_visuals=VISUAL_ENRICHMENT,
            ocr_score_threshold=OCR_SCORE_THRESHOLD,
        )

        rag_ready_text = build_rag_ready_text(
            parsed,
            include_table_blocks=True,
            include_visual_hints=INCLUDE_VISUAL_HINTS_IN_RAG_TEXT,
        )

        record = {
            "document_id": f"{file_path.suffix.lower().lstrip('.')}_{file_path.stem}",
            "source_file_name": file_path.name,
            "source_path": str(file_path),
            "source_extension": file_path.suffix.lower(),
            "text": parsed["text"],
            "rag_ready_text": rag_ready_text,
            "metadata": parsed["metadata"],
            "parser_info": parsed["parser_info"],
            "artifacts": parsed["artifacts"],
        }
        processed_records.append(record)

        summary_rows.append(
            {
                "file_name": file_path.name,
                "ext": file_path.suffix.lower(),
                "parser_used": parsed["parser_info"]["parser_used"],
                "text_len": len(parsed["text"]),
                "rag_ready_text_len": len(rag_ready_text),
                "project_name": parsed["metadata"].get(COL_PROJECT_NAME),
                "agency": parsed["metadata"].get(COL_AGENCY),
                "budget": parsed["metadata"].get(COL_BUDGET),
                "posted_at": parsed["metadata"].get(COL_POSTED_AT),
                "bid_start": parsed["metadata"].get(COL_BID_START),
                "bid_end": parsed["metadata"].get(COL_BID_END),
                "table_markers": parsed["metadata"].get("table_markers"),
                "figure_markers": parsed["metadata"].get("figure_markers"),
                "chart_markers": parsed["metadata"].get("chart_markers"),
                "bin_count": parsed["metadata"].get("bin_count"),
                "pdf_table_count_first_pages": parsed["metadata"].get("pdf_table_count_first_pages"),
                "enriched_table_count": parsed["metadata"].get("enriched_table_count"),
                "visual_candidate_count": parsed["metadata"].get("visual_candidate_count"),
                "ocr_candidate_count": parsed["metadata"].get("ocr_candidate_count"),
                "visual_priority_max_score": parsed["metadata"].get("visual_priority_max_score"),
            }
        )

        for candidate in parsed["artifacts"].get("ocr_queue", []):
            ocr_rows.append(
                {
                    "file_name": file_path.name,
                    "ext": file_path.suffix.lower(),
                    "project_name": parsed["metadata"].get(COL_PROJECT_NAME),
                    "agency": parsed["metadata"].get(COL_AGENCY),
                    **candidate,
                }
            )

        if idx % 10 == 0 or idx == len(all_files):
            print(f"[{idx}/{len(all_files)}] processed: {file_path.name}")

    except Exception as exc:
        error_rows.append(
            {
                "file_name": file_path.name,
                "source_path": str(file_path),
                "ext": file_path.suffix.lower(),
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )
        print(f"[ERROR] {file_path.name}: {type(exc).__name__} - {exc}")

with DOCUMENTS_JSONL_PATH.open("w", encoding="utf-8") as fp:
    for record in processed_records:
        fp.write(json.dumps(record, ensure_ascii=False) + "\n")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV_PATH, index=False, encoding="utf-8-sig")

with OCR_QUEUE_JSONL_PATH.open("w", encoding="utf-8") as fp:
    for row in ocr_rows:
        fp.write(json.dumps(row, ensure_ascii=False) + "\n")

if error_rows:
    with ERRORS_JSONL_PATH.open("w", encoding="utf-8") as fp:
        for row in error_rows:
            fp.write(json.dumps(row, ensure_ascii=False) + "\n")
elif ERRORS_JSONL_PATH.exists():
    ERRORS_JSONL_PATH.unlink()

RUN_CONFIG_PATH.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

print("전처리 완료 문서 수:", len(processed_records))
print("OCR 후보 행 수:", len(ocr_rows))
print("에러 행 수:", len(error_rows))
print("저장된 documents jsonl:", DOCUMENTS_JSONL_PATH)
print("저장된 summary csv:", SUMMARY_CSV_PATH)
print("저장된 OCR queue jsonl:", OCR_QUEUE_JSONL_PATH)
print("저장된 run config:", RUN_CONFIG_PATH)
if error_rows:
    print("저장된 errors jsonl:", ERRORS_JSONL_PATH)

display(summary_df.head(10))


## 7. 최종 해석

이 전처리 노트북은 의도적으로 2단계 구조로 설계했다.

1단계 baseline:

- HWP 기본 파서: `hwp5txt`
- HWP 폴백: `olefile + PrvText`
- PDF 기본 파서: `pdfplumber`
- PDF 폴백: `PyMuPDF`
- 표, 그림, 차트, 바이너리 자산 관련 구조 메타데이터를 유지

2단계 enrichment:

- 추출 가능한 표는 텍스트 블록으로 변환
- 가치가 높은 시각자료는 자동 shortlist
- OCR은 shortlist 된 후보에만 적용
- 사람 검토는 shortlist 되거나 애매한 경우에만 사용

RAG 관점의 시사점:

- 표 블록은 구체적 사실 정보가 많으므로 retrieval corpus에 함께 넣는 것이 좋다.
- 시각자료 질문에는 단순 marker 보존만으로 충분하지 않다.
- 아키텍처 다이어그램, 프로세스 흐름도, 화면 예시, 차트 등은 선택적 OCR 대상이 되어야 한다.
